In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
baseline_blocks = gdf = pd.read_pickle("./data/traning_data/blocks.pickle")
baseline_blocks.head()

In [ ]:
from blocksnet.analysis.indicators import calculate_density_indicators 

# 1. Если CRS географический (deg), переведём в метрический (например, Меркатор)
if baseline_blocks.crs.is_geographic:
    blocks = baseline_blocks.to_crs(epsg=3857)

# 2. Считаем площадь каждого квартала в м²
baseline_blocks['site_area'] = baseline_blocks.geometry.area

blocks_df = calculate_density_indicators(baseline_blocks[[
    'site_area',
    'footprint_area', 
    'build_floor_area',
    'living_area',
    'non_living_area'
]])
blocks_df

In [ ]:
baseline_blocks.loc[:, blocks_df.columns] = blocks_df
baseline_blocks.head()

In [ ]:
from blocksnet.analysis.morphotypes import get_strelka_morphotypes


blocks_df = get_strelka_morphotypes(baseline_blocks)
blocks_df

In [ ]:
baseline_blocks.loc[:, blocks_df.columns] = blocks_df
baseline_blocks.head()

In [ ]:
from blocksnet.relations import generate_adjacency_graph

adjacency_graph = generate_adjacency_graph(baseline_blocks)


accessibility_matrix = pd.read_pickle('./data/traning_data/acc_mx.pickle')
accessibility_matrix.head()

In [ ]:
from blocksnet.analysis.network.accessibility import area_accessibility

area_acc_df = area_accessibility(accessibility_matrix, baseline_blocks)
baseline_blocks= baseline_blocks.join(area_acc_df)

In [ ]:
cols_to_drop = [col for col in baseline_blocks.columns if col.startswith('capacity')]
baseline_blocks = baseline_blocks.drop(columns=cols_to_drop)
# cols_to_drop = [col for col in blocks.columns if col.startswith('count')]
# blocks = blocks.drop(columns='cluster')
# blocks = blocks.drop(columns=cols_to_drop)

baseline_blocks.columns

In [ ]:
# cols = [
#     'residential','business','recreation','industrial','transport','special',
#     'agriculture','land_use','share','footprint_area','build_floor_area',
#     'living_area','non_living_area','population','site_area','fsi','gsi',
#     'mxi','l','morphotype','area_accessibility', 'geometry'
# ]
# baseline_blocks = baseline_blocks[cols]
# baseline_blocks.head()

In [ ]:
from catboost import CatBoostRegressor

from urbanomy.methods.land_value_modeling import LandPriceEstimator
model = CatBoostRegressor()

model.load_model('./data/models/land_value_catboost_22_12.cbm')

estimator = LandPriceEstimator(
    model=model,
    blocks=baseline_blocks, #or blocks_198
    use_service_features=True, 
)
blocks_pred = estimator.predict()
blocks_pred.head()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# 1) Цена за сотку (100 м²)
blocks_pred["land_value_per_100m2"] = blocks_pred["land_value"] / blocks_pred["site_area"] * 100

# 2) Заменяем inf на NaN
blocks_pred = blocks_pred.replace([np.inf, -np.inf], np.nan)
blocks_pred = blocks_pred.fillna(0)

blocks_pred['log_total_price'] = np.log1p(blocks_pred['land_value'])
# 3) Удаляем выбросы: оставляем данные до 99-го перцентиля
p99 = blocks_pred["land_value_per_100m2"].quantile(0.98)
blocks_clean = blocks_pred[blocks_pred["land_value_per_100m2"] <= p99].copy()

# p99 = blocks_clean["land_value"].quantile(0.98)
# blocks_clean = blocks_clean[blocks_clean["land_value"] <= p99].copy()

print(blocks_clean["land_value_per_100m2"].describe())

# 4) Гистограмма после очистки
plt.figure()
blocks_clean["log_total_price"].dropna().hist(bins=50)
plt.xlabel("Цена за сотку (руб.)")
plt.ylabel("Частота")
# plt.title("Распределение цены за сотку")
plt.show()

# 5) Box-plot после очистки
plt.figure()
plt.boxplot(blocks_clean["log_total_price"].dropna(), vert=False)
# plt.xlabel("Цена за сотку (руб.)")
plt.title("Box-plot цены за сотку")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value_per_100m2',
    legend=True,
    figsize=(20,20),
    cmap='coolwarm',
    edgecolor='black',   # <-- цвет границы
    linewidth=0.2        # <-- толщина границы
).set_axis_off()
plt.title('Карта стоимости землельных участков за сотку (руб.)', fontsize=16)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

blocks_clean.plot(
    column='land_value',
    legend=True,
    figsize=(20,20),
    cmap='coolwarm',
    edgecolor='black',   # <-- цвет границы
    linewidth=0.2        # <-- толщина границы
).set_axis_off()
plt.title('Карта стоимости земельных участков (руб.)', fontsize=16)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
baseline_blocks["id"] = baseline_blocks.index

target_idx = 723


fig, ax = plt.subplots(figsize=(25, 35))
baseline_blocks.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.6)  # фон
baseline_blocks.loc[baseline_blocks["id"]==target_idx].plot(
    ax=ax, color="none", edgecolor="gold", linewidth=5.5
)
ax.set_title("Изменяемый квартал в Гатчине")
ax.axis("off")
plt.show()

In [ ]:
from urbanomy.methods.land_value_modeling import (
    ScenarioTEPModifier,
    plot_scenario_impact,
    transfer_baseline_prices,
)

blocks_before = baseline_blocks.copy()

changes = {
    # Структура использования территории
    "land_use": "LandUse.RESIDENTIAL",
    "residential": 0.8,
    "business": 0.2,
    "recreation": 0.0,
    "industrial": 0.0,
    "transport": 0.0,
    "special": 0.0,
    "agriculture": 0.0,
    "share": 1.0,

    "footprint_area": 48170.0451624,    # ≈ 60% от site_area
    "build_floor_area": 481700.451624,  # FSI ≈ 6.0
    "living_area": 337190.3161368,      # ≈ 70% 
    "non_living_area": 144510.1354872,  # ≈ 30% 
    "population": 5537,                # 


    "fsi": 6.0,                         # build_floor_area / site_area
    "gsi": 0.60,                        # footprint_area / site_area
    "l": 10.0,                          # build_floor_area / footprint_area (этажность)
                    # build_floor_area / footprint_area (этажность)

    # Морфотип – высотный дом
    "morphotype": "high-rise residential",
    }


modifier = ScenarioTEPModifier(blocks_before)
blocks_after = modifier.apply(target_idx, changes)

# 1) Предсказание стоимости до/после
before_estimator = LandPriceEstimator(model=model, blocks=blocks_before, use_service_features=True,)
after_estimator = LandPriceEstimator(model=model, blocks=blocks_after, use_service_features=True,)

blocks_before_pred = before_estimator.predict()
blocks_before_pred["land_value_per_100m2"] = blocks_before_pred["land_value"] / blocks_before_pred["site_area"] * 100

blocks_after_pred = after_estimator.predict()
blocks_after_pred["land_value_per_100m2"] = blocks_after_pred["land_value"] / blocks_after_pred["site_area"] * 100

blocks_full_value = transfer_baseline_prices(
    after_blocks=blocks_after_pred,
    before_blocks=blocks_before_pred,
    scenario_mode=False,
)

blocks_full_value = blocks_full_value.replace([np.inf, -np.inf], np.nan)
blocks_full_value = blocks_full_value.fillna(0)

# 3) Только визуализация и статистика по процентным изменениям
scenario_result = plot_scenario_impact(
    blocks=blocks_full_value,
    target_idx=target_idx,          
    target_id_column="id",   # если колонка называется иначе — укажи здесь
    figsize=(25, 35),
)